# Notebook 1 - Point of Interest Dataset

## BT Geo-Temporal Population Project

This notebook creates the Point of Interest (POI) dataset used throughout the project.

The purpose of the POI dataset is to provide geographic information about important locations within and around the BT study area. This information is later combined with the BT network data so that unusual network activity can be interpreted in relation to nearby real-world locations.

POIs are downloaded from OpenStreetMap using the Overpass API.

The download area covers the complete BT study area together with an additional 5 km buffer around its boundary. Therefore, POIs are collected both inside the BT study area and within 5 km of its edge.

The notebook currently downloads 24 selected POI categories. These include locations such as transport facilities, accommodation, attractions, sports facilities and other places that may help explain patterns in the BT network data.

The notebook also contains additional optional POI categories that are not currently selected. These can easily be enabled if more types of locations are required in the future.

Each selected POI category is downloaded separately and saved so that successful downloads can be reused when the notebook is run again.

The downloaded OpenStreetMap data are then cleaned and standardised, converted to representative point locations and projected to UTM Zone 30N (EPSG:32630). All 24 selected categories are then combined into one final POI dataset.

The final output file is:

`data/processed_poi_locations/BT_POI_Dataset.gpkg`

using the layer:

`combined_pois`

The final dataset contains:

- `poi_id`
- `poi_category`
- `poi_name`
- `osm_element_type`
- `osm_id`
- `poi_source`
- `download_buffer_metres`
- `geometry`

In `2_Cleaning_and_Feature_Engineering` (Notebook 2), this POI dataset is combined with the BT network data. Each row of the BT data contains network measurements associated with a particular BT grid location. For every BT grid location represented in the data, the notebook calculates the distance to the nearest POI from each of the POI categories created here. These distances are added to the BT observations as new geographic features, allowing the later models to identify unusual network activity and investigate the types of real-world locations nearby.

`2_Cleaning_and_Feature_Engineering` (Notebook 2) then produces the final prepared daily and hourly datasets. 

The daily dataset, including its POI information, is used by `3_Model_1` (Notebook 3), while the hourly dataset is used by `4_Model_2` (Notebook 4).

This means that `1_POI_Dataset_Overpass` (Notebook 1) provides the geographic context that is carried through `2_Cleaning_and_Feature_Engineering` (Notebook 2) and into the two modelling notebooks, `3_Model_1` (Notebook 3) and `4_Model_2` (Notebook 4).

# Notebook Roadmap

| Section | Cells | Purpose |
|---|---:|---|
| 1. Project Setup | 1–2 | Import packages and configure the project folders |
| 2. Study and Download Areas | 3–4 | Create the original BT area and its 5 km POI download buffer |
| 3. POI Configuration | 5–7 | Configure Overpass and define the active and optional POI categories |
| 4. Download Workflow | 8–9 | Create the download functions and process all configured categories |
| 5. Download Verification | 10 | Verify every configured category file |
| 6. POI Dataset Preparation | 11–12 | Standardise and combine all configured POI categories |
| 7. `2_Cleaning_and_Feature_Engineering` (Notebook 2) Compatibility | 13–14 | Convert the combined POI dataset to the schema required by `2_Cleaning_and_Feature_Engineering` (Notebook 2) and verify compatibility |
| 8. Final Output | 15–16 | Save, reload and fully verify the final POI input for `2_Cleaning_and_Feature_Engineering` (Notebook 2) |

The notebook contains 16 numbered code cells.

A short explanation markdown cell follows every numbered code cell.

# Python Package Setup

This cell checks that the Python packages required by the project are available.

Packages that are already installed are left unchanged. If a required package is missing, it is installed automatically into the Python environment used by this notebook.

This allows the notebook to run on another computer without requiring the packages to be installed manually beforehand.

In [1]:
# ================================================================
# CHECK AND INSTALL REQUIRED PYTHON PACKAGES
# ================================================================

import importlib.util
import subprocess
import sys


required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "duckdb": "duckdb",
    "geopandas": "geopandas",
    "fiona": "fiona",
    "shapely": "shapely",
    "osmnx": "osmnx",
    "pyproj": "pyproj",
    "folium": "folium",
    "IPython": "ipython"
}


print("=" * 80)
print("CHECK AND INSTALL REQUIRED PYTHON PACKAGES")
print("=" * 80)


installed_packages = []
missing_packages = []


for import_name, package_name in required_packages.items():

    if importlib.util.find_spec(import_name) is None:

        missing_packages.append(
            package_name
        )

    else:

        installed_packages.append(
            package_name
        )


print(
    f"\nPackages already available: "
    f"{len(installed_packages):,}"
)

for package_name in installed_packages:

    print(
        f"  - {package_name}"
    )


if missing_packages:

    print(
        f"\nPackages requiring installation: "
        f"{len(missing_packages):,}"
    )

    for package_name in missing_packages:

        print(
            f"  - {package_name}"
        )


    print(
        "\nInstalling missing packages..."
    )


    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            *missing_packages
        ]
    )


    print(
        "\nMissing packages were installed successfully."
    )

else:

    print(
        "\nAll required packages are already installed."
    )


print(
    "\nPython package setup complete."
)

CHECK AND INSTALL REQUIRED PYTHON PACKAGES

Packages already available: 15
  - numpy
  - pandas
  - pyarrow
  - matplotlib
  - seaborn
  - scikit-learn
  - joblib
  - duckdb
  - geopandas
  - fiona
  - shapely
  - osmnx
  - pyproj
  - folium
  - ipython

All required packages are already installed.

Python package setup complete.


# 1. Project Setup

This section imports the Python packages required for the POI workflow and configures the project folders used for the individual Overpass downloads and the final combined POI dataset.

The final POI dataset is saved in the project’s processed data folder, ready to be loaded and combined with the BT network data in `2_Cleaning_and_Feature_Engineering`.

In [2]:
# ================================================================
# CELL 1 - IMPORT PYTHON PACKAGES
# ================================================================

from pathlib import Path

import time
import warnings

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd

from IPython.display import display
from shapely.geometry import box


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    220
)


warnings.filterwarnings(
    "default"
)


print("=" * 80)
print("CELL 1 - IMPORT PYTHON PACKAGES")
print("=" * 80)


package_versions = pd.DataFrame(
    {
        "Package": [
            "pandas",
            "GeoPandas",
            "NumPy",
            "OSMnx"
        ],
        "Version": [
            pd.__version__,
            gpd.__version__,
            np.__version__,
            ox.__version__
        ]
    }
)


display(
    package_versions
)


print(
    "\nThe required Python packages were "
    "imported successfully."
)

CELL 1 - IMPORT PYTHON PACKAGES


,Package,Version
0,pandas,2.2.3
1,GeoPandas,1.1.4
2,NumPy,2.1.3
3,OSMnx,2.1.0



The required Python packages were imported successfully.


## What Cell 1 Does

This cell imports the Python packages required for the POI workflow.

The packages provide tools for:

- file and folder management;
- timing and controlled pauses;
- tabular data processing;
- spatial data processing;
- coordinate transformations;
- geometric operations;
- HTTP requests to the Overpass API;
- displaying notebook tables.

These packages are used throughout the remaining notebook.

In [3]:
# ================================================================
# CELL 2 - CONFIGURE PROJECT FOLDERS
# ================================================================

EXPECTED_PROJECT_NAME = (
    "BT_Dissertation_AL"
)


def find_project_folder():
    """
    Locate the main BT dissertation project folder.
    """

    current_folder = Path.cwd().resolve()

    candidate_folders = [
        current_folder,
        *current_folder.parents
    ]


    for candidate_folder in candidate_folders:

        if candidate_folder.name == EXPECTED_PROJECT_NAME:

            return candidate_folder


        if (
            candidate_folder
            / "data"
        ).exists():

            return candidate_folder


    raise FileNotFoundError(
        "The BT dissertation project folder could "
        "not be located."
    )


PROJECT_FOLDER = find_project_folder()


DATA_FOLDER = (
    PROJECT_FOLDER
    / "data"
)


POI_DOWNLOAD_FOLDER = (
    DATA_FOLDER
    / "poi_downloads_5km_buffer"
)


PROCESSED_POI_FOLDER = (
    DATA_FOLDER
    / "processed_poi_locations"
)


FINAL_POI_FILE = (
    PROCESSED_POI_FOLDER
    / "BT_POI_Dataset.gpkg"
)


FINAL_POI_LAYER = (
    "combined_pois"
)


POI_DOWNLOAD_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


PROCESSED_POI_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


folder_summary = pd.DataFrame(
    {
        "Purpose": [
            "Project folder",
            "Data folder",
            "Buffered category downloads",
            "Processed POI folder",
            "Final POI GeoPackage",
            "Final POI layer"
        ],
        "Location": [
            str(
                PROJECT_FOLDER
            ),
            str(
                DATA_FOLDER
            ),
            str(
                POI_DOWNLOAD_FOLDER
            ),
            str(
                PROCESSED_POI_FOLDER
            ),
            str(
                FINAL_POI_FILE
            ),
            FINAL_POI_LAYER
        ]
    }
)


print("=" * 80)
print("CELL 2 - CONFIGURE PROJECT FOLDERS")
print("=" * 80)


display(
    folder_summary
)


assert PROJECT_FOLDER.exists()

assert DATA_FOLDER.exists()

assert POI_DOWNLOAD_FOLDER.exists()

assert PROCESSED_POI_FOLDER.exists()

assert POI_DOWNLOAD_FOLDER.resolve() != (
    PROCESSED_POI_FOLDER.resolve()
)


print(
    "\nThe POI project folders were configured "
    "successfully."
)

CELL 2 - CONFIGURE PROJECT FOLDERS


,Purpose,Location
0,Project folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
1,Data folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
2,Buffered category downloads,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
3,Processed POI folder,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
4,Final POI GeoPackage,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
5,Final POI layer,combined_pois



The POI project folders were configured successfully.


## What Cell 2 Does

This cell locates the main dissertation project folder and creates the folders used by the POI workflow.

Individual category downloads are stored in:

`data/poi_downloads_5km_buffer`

Each successful category file remains in this folder so that it can be validated and reused when the notebook is run again.

The completed POI input for `2_Cleaning_and_Feature_Engineering` (Notebook 2) is saved as:

`data/processed_poi_locations/BT_POI_Dataset.gpkg`

using the layer:

`combined_pois`

This keeps the final filename, folder and layer consistent with the downstream POI input used by `2_Cleaning_and_Feature_Engineering` (Notebook 2).

# 2. Study and Download Areas

This section creates the original BT study area and a separate 5 km buffered area used only for downloading nearby POIs.

The BT study boundary itself is not expanded. The buffer ensures that relevant POIs immediately outside the study boundary can still be included when distance-based POI features are calculated later.

In [4]:
# ================================================================
# CELL 3 - CREATE THE ORIGINAL BT STUDY AREA
# ================================================================

STUDY_XMIN = (
    530_000
)

STUDY_YMIN = (
    5_590_000
)

STUDY_XMAX = (
    668_000
)

STUDY_YMAX = (
    5_650_000
)


PROJECTED_CRS = (
    "EPSG:32630"
)


GEOGRAPHIC_CRS = (
    "EPSG:4326"
)


original_study_geometry = box(
    STUDY_XMIN,
    STUDY_YMIN,
    STUDY_XMAX,
    STUDY_YMAX
)


study_area = gpd.GeoDataFrame(
    {
        "area_name": [
            "Original BT study area"
        ]
    },
    geometry=[
        original_study_geometry
    ],
    crs=PROJECTED_CRS
)


study_area_width_km = (
    STUDY_XMAX
    -
    STUDY_XMIN
) / 1_000


study_area_height_km = (
    STUDY_YMAX
    -
    STUDY_YMIN
) / 1_000


study_area_square_km = (
    study_area.geometry.area.iloc[0]
    /
    1_000_000
)


study_area_summary = pd.DataFrame(
    {
        "Measure": [
            "Projected CRS",
            "Minimum easting",
            "Minimum northing",
            "Maximum easting",
            "Maximum northing",
            "Width (km)",
            "Height (km)",
            "Area (square km)"
        ],
        "Value": [
            str(
                study_area.crs
            ),
            STUDY_XMIN,
            STUDY_YMIN,
            STUDY_XMAX,
            STUDY_YMAX,
            study_area_width_km,
            study_area_height_km,
            study_area_square_km
        ]
    }
)


print("=" * 80)
print("CELL 3 - CREATE THE ORIGINAL BT STUDY AREA")
print("=" * 80)


display(
    study_area_summary
)


assert study_area.crs.to_epsg() == 32630

assert study_area.geometry.is_valid.all()

assert study_area.geometry.area.iloc[0] > 0

assert study_area_width_km == 138

assert study_area_height_km == 60


print(
    "\nThe original BT study area was created "
    "successfully."
)

CELL 3 - CREATE THE ORIGINAL BT STUDY AREA


,Measure,Value
0,Projected CRS,EPSG:32630
1,Minimum easting,530000
2,Minimum northing,5590000
3,Maximum easting,668000
4,Maximum northing,5650000
5,Width (km),138.0
6,Height (km),60.0
7,Area (square km),8280.0



The original BT study area was created successfully.


## What Cell 3 Does

This cell recreates the original BT study area from the supplied UTM Zone 30N coordinates.

The study bounds are:

- `xmin = 530000`
- `ymin = 5590000`
- `xmax = 668000`
- `ymax = 5650000`

The study area is created in EPSG:32630, which uses metre-based coordinates.

This remains the official BT analysis area. The later 5 km buffer does not change these original study bounds.

In [5]:
# ================================================================
# CELL 4 - CREATE THE 5 KM POI DOWNLOAD BUFFER
# ================================================================

POI_DOWNLOAD_BUFFER_METRES = (
    5_000
)


buffered_download_area = (
    study_area.copy()
)


buffered_download_area[
    "area_name"
] = (
    "BT study area with 5 km POI download buffer"
)


buffered_download_area[
    "geometry"
] = (
    buffered_download_area.geometry.buffer(
        POI_DOWNLOAD_BUFFER_METRES
    )
)


buffered_download_area_wgs84 = (
    buffered_download_area.to_crs(
        GEOGRAPHIC_CRS
    )
)


(
    buffered_left,
    buffered_bottom,
    buffered_right,
    buffered_top
) = buffered_download_area_wgs84.total_bounds


OVERPASS_BBOX = (
    float(
        buffered_left
    ),
    float(
        buffered_bottom
    ),
    float(
        buffered_right
    ),
    float(
        buffered_top
    )
)


buffered_projected_bounds = (
    buffered_download_area.total_bounds
)


buffer_summary = pd.DataFrame(
    {
        "Measure": [
            "Original study minimum easting",
            "Original study minimum northing",
            "Original study maximum easting",
            "Original study maximum northing",
            "Buffer distance (metres)",
            "Buffered minimum easting",
            "Buffered minimum northing",
            "Buffered maximum easting",
            "Buffered maximum northing",
            "Overpass left longitude",
            "Overpass bottom latitude",
            "Overpass right longitude",
            "Overpass top latitude"
        ],
        "Value": [
            STUDY_XMIN,
            STUDY_YMIN,
            STUDY_XMAX,
            STUDY_YMAX,
            POI_DOWNLOAD_BUFFER_METRES,
            float(
                buffered_projected_bounds[0]
            ),
            float(
                buffered_projected_bounds[1]
            ),
            float(
                buffered_projected_bounds[2]
            ),
            float(
                buffered_projected_bounds[3]
            ),
            OVERPASS_BBOX[0],
            OVERPASS_BBOX[1],
            OVERPASS_BBOX[2],
            OVERPASS_BBOX[3]
        ]
    }
)


print("=" * 80)
print("CELL 4 - CREATE THE 5 KM POI DOWNLOAD BUFFER")
print("=" * 80)


display(
    buffer_summary
)


assert buffered_download_area.crs.to_epsg() == 32630

assert buffered_download_area_wgs84.crs.to_epsg() == 4326

assert buffered_download_area.geometry.is_valid.all()

assert buffered_download_area.geometry.contains(
    study_area.geometry.iloc[0]
).all()

assert buffered_projected_bounds[0] == (
    STUDY_XMIN
    -
    POI_DOWNLOAD_BUFFER_METRES
)

assert buffered_projected_bounds[1] == (
    STUDY_YMIN
    -
    POI_DOWNLOAD_BUFFER_METRES
)

assert buffered_projected_bounds[2] == (
    STUDY_XMAX
    +
    POI_DOWNLOAD_BUFFER_METRES
)

assert buffered_projected_bounds[3] == (
    STUDY_YMAX
    +
    POI_DOWNLOAD_BUFFER_METRES
)

assert OVERPASS_BBOX[0] < OVERPASS_BBOX[2]

assert OVERPASS_BBOX[1] < OVERPASS_BBOX[3]


print(
    "\nThe 5 km POI download buffer was created "
    "successfully."
)

CELL 4 - CREATE THE 5 KM POI DOWNLOAD BUFFER


,Measure,Value
0,Original study minimum easting,5.300000e+05
1,Original study minimum northing,5.590000e+06
2,Original study maximum easting,6.680000e+05
3,Original study maximum northing,5.650000e+06
4,Buffer distance (metres),5.000000e+03
5,Buffered minimum easting,5.250000e+05
6,Buffered minimum northing,5.585000e+06
7,Buffered maximum easting,6.730000e+05
8,Buffered maximum northing,5.655000e+06
9,Overpass left longitude,-2.647781e+00



The 5 km POI download buffer was created successfully.


## What Cell 4 Does

This cell creates a 5 km buffer around the original BT study area.

Because EPSG:32630 uses metres, the buffer distance is defined as:

`5000 metres`

The buffered polygon is transformed to WGS84 (EPSG:4326) because Overpass queries use longitude and latitude coordinates.

The resulting buffered bounding box is used only for downloading POIs. It does not replace or enlarge the original BT analysis area.

# 3. POI Configuration

This section configures the OpenStreetMap Overpass workflow and defines the POI categories that will be downloaded.

The category definitions determine the exact OpenStreetMap tags requested for each POI type. The workflow remains dynamic: the number of active categories is calculated from the current configuration rather than being permanently fixed in the later validation code.

In [6]:
# ================================================================
# CELL 5 - CONFIGURE THE MAIN OVERPASS SERVER
# ================================================================

OVERPASS_MAIN_SERVER = (
    "https://overpass-api.de/api"
)


OVERPASS_TIMEOUT_SECONDS = (
    300
)


ox.settings.overpass_url = (
    OVERPASS_MAIN_SERVER
)


ox.settings.requests_timeout = (
    OVERPASS_TIMEOUT_SECONDS
)


ox.settings.use_cache = True

ox.settings.log_console = True

ox.settings.overpass_rate_limit = True


overpass_settings_summary = pd.DataFrame(
    {
        "Setting": [
            "Overpass server",
            "Timeout (seconds)",
            "OSMnx cache enabled",
            "Console logging enabled",
            "Overpass rate limiting enabled",
            "Notebook download calls per category"
        ],
        "Value": [
            ox.settings.overpass_url,
            ox.settings.requests_timeout,
            ox.settings.use_cache,
            ox.settings.log_console,
            ox.settings.overpass_rate_limit,
            1
        ]
    }
)


print("=" * 80)
print("CELL 5 - CONFIGURE THE MAIN OVERPASS SERVER")
print("=" * 80)


display(
    overpass_settings_summary
)


assert ox.settings.overpass_url == (
    OVERPASS_MAIN_SERVER
)

assert ox.settings.requests_timeout == (
    OVERPASS_TIMEOUT_SECONDS
)


print(
    "\nThe main Overpass server was configured "
    "successfully."
)

CELL 5 - CONFIGURE THE MAIN OVERPASS SERVER


,Setting,Value
0,Overpass server,https://overpass-api.de/api
1,Timeout (seconds),300
2,OSMnx cache enabled,True
3,Console logging enabled,True
4,Overpass rate limiting enabled,True
5,Notebook download calls per category,1



The main Overpass server was configured successfully.


## What Cell 5 Does

This cell configures the Overpass API endpoints and the settings used when requesting OpenStreetMap data.

Multiple endpoints are available so that another endpoint can be attempted if one is temporarily unavailable.

The cell also defines the request timeout, retry behaviour and pause settings used by the download workflow.

In [7]:
# ================================================================
# CELL 6 - DEFINE THE ACTIVE POINT OF INTEREST CATEGORIES
# ================================================================

POI_CONFIG = {
    "stadiums": {
        "label": "Stadiums",
        "tags": {
            "leisure": "stadium"
        }
    },

    "sport_centres": {
        "label": "Sport Centres",
        "tags": {
            "leisure": "sports_centre"
        }
    },

    "holiday_parks": {
        "label": "Holiday Parks",
        "tags": {
            "tourism": "caravan_site"
        }
    },

    "theme_parks": {
        "label": "Theme Parks",
        "tags": {
            "tourism": "theme_park"
        }
    },

    "zoos": {
        "label": "Zoos",
        "tags": {
            "tourism": "zoo"
        }
    },

    "railway_stations": {
        "label": "Railway Stations",
        "tags": {
            "railway": "station"
        }
    },

    "bus_stations": {
        "label": "Bus Stations",
        "tags": {
            "amenity": "bus_station"
        }
    },

    "bus_stops": {
        "label": "Bus Stops",
        "tags": {
            "highway": "bus_stop"
        }
    },

    "airports": {
        "label": "Airports",
        "tags": {
            "aeroway": "aerodrome"
        }
    },

    "hospitals": {
        "label": "Hospitals",
        "tags": {
            "amenity": "hospital"
        }
    },

    "schools": {
        "label": "Schools",
        "tags": {
            "amenity": "school"
        }
    },

    "universities": {
        "label": "Universities",
        "tags": {
            "amenity": "university"
        }
    },

    "shopping_malls": {
        "label": "Shopping Malls",
        "tags": {
            "shop": "mall"
        }
    },

    "supermarkets": {
        "label": "Supermarkets",
        "tags": {
            "shop": "supermarket"
        }
    },

    "offices": {
        "label": "Offices",
        "tags": {
            "office": True
        }
    },

    "beaches": {
        "label": "Beaches",
        "tags": {
            "natural": "beach"
        }
    },

    "attractions": {
        "label": "Attractions",
        "tags": {
            "tourism": "attraction"
        }
    },

    "hotels": {
        "label": "Hotels",
        "tags": {
            "tourism": "hotel"
        }
    },

    "pubs": {
        "label": "Pubs",
        "tags": {
            "amenity": "pub"
        }
    },

    "restaurants": {
        "label": "Restaurants",
        "tags": {
            "amenity": "restaurant"
        }
    },

    "cafes": {
        "label": "Cafes",
        "tags": {
            "amenity": "cafe"
        }
    },

    "fast_foods": {
        "label": "Fast Foods",
        "tags": {
            "amenity": "fast_food"
        }
    },

    "parks": {
        "label": "Parks",
        "tags": {
            "leisure": "park"
        }
    },

    "gyms": {
        "label": "Gyms",
        "tags": {
            "leisure": "fitness_centre"
        }
    }
}


CONFIGURED_POI_CATEGORY_COUNT = len(
    POI_CONFIG
)


CONFIGURED_POI_TYPES = list(
    POI_CONFIG.keys()
)


poi_category_table = pd.DataFrame(
    [
        {
            "Category Number": category_number,
            "POI Type": poi_type,
            "POI Label": config[
                "label"
            ],
            "OpenStreetMap Tags": str(
                config[
                    "tags"
                ]
            )
        }
        for category_number, (
            poi_type,
            config
        ) in enumerate(
            POI_CONFIG.items(),
            start=1
        )
    ]
)


print("=" * 80)
print("CELL 6 - DEFINE THE ACTIVE POINT OF INTEREST CATEGORIES")
print("=" * 80)


print(
    f"\nConfigured POI categories: "
    f"{CONFIGURED_POI_CATEGORY_COUNT:,}"
)


display(
    poi_category_table
)


assert CONFIGURED_POI_CATEGORY_COUNT > 0

assert len(
    CONFIGURED_POI_TYPES
) == CONFIGURED_POI_CATEGORY_COUNT

assert len(
    set(
        CONFIGURED_POI_TYPES
    )
) == CONFIGURED_POI_CATEGORY_COUNT

assert poi_category_table[
    "POI Type"
].is_unique

assert poi_category_table[
    "POI Label"
].notna().all()


print(
    "\nThe active Point of Interest categories "
    "were defined successfully."
)

CELL 6 - DEFINE THE ACTIVE POINT OF INTEREST CATEGORIES

Configured POI categories: 24


,Category Number,POI Type,POI Label,OpenStreetMap Tags
0,1,stadiums,Stadiums,{'leisure': 'stadium'}
1,2,sport_centres,Sport Centres,{'leisure': 'sports_centre'}
2,3,holiday_parks,Holiday Parks,{'tourism': 'caravan_site'}
3,4,theme_parks,Theme Parks,{'tourism': 'theme_park'}
4,5,zoos,Zoos,{'tourism': 'zoo'}
5,6,railway_stations,Railway Stations,{'railway': 'station'}
6,7,bus_stations,Bus Stations,{'amenity': 'bus_station'}
7,8,bus_stops,Bus Stops,{'highway': 'bus_stop'}
8,9,airports,Airports,{'aeroway': 'aerodrome'}
9,10,hospitals,Hospitals,{'amenity': 'hospital'}



The active Point of Interest categories were defined successfully.


## What Cell 6 Does

This cell defines the active POI categories and their OpenStreetMap tag rules.

Each category is associated with the exact OpenStreetMap key-value combination required by the project.

These definitions control which OpenStreetMap features are requested from Overpass.

The active category count is calculated dynamically from `POI_CONFIG`, so later validation automatically follows the current configuration.

In [8]:
# ================================================================
# CELL 7 - OPTIONAL POINT OF INTEREST CATEGORY CATALOGUE
# ================================================================

# The examples below are not active and will not be downloaded.
#
# To add one of these categories:
#
# 1. Copy its complete entry.
# 2. Paste the entry inside POI_CONFIG in Cell 6.
# 3. Ensure that the category key is unique.
# 4. Rerun the notebook.
#
# The later cells will automatically download, verify,
# standardise and combine the additional category.


OPTIONAL_POI_CATEGORY_CATALOGUE = {

    # "museums": {
    #     "label": "Museums",
    #     "tags": {
    #         "tourism": "museum"
    #     }
    # },

    # "cinemas": {
    #     "label": "Cinemas",
    #     "tags": {
    #         "amenity": "cinema"
    #     }
    # },

    # "theatres": {
    #     "label": "Theatres",
    #     "tags": {
    #         "amenity": "theatre"
    #     }
    # },

    # "libraries": {
    #     "label": "Libraries",
    #     "tags": {
    #         "amenity": "library"
    #     }
    # },

    # "colleges": {
    #     "label": "Colleges",
    #     "tags": {
    #         "amenity": "college"
    #     }
    # },

    # "kindergartens": {
    #     "label": "Kindergartens",
    #     "tags": {
    #         "amenity": "kindergarten"
    #     }
    # },

    # "doctors": {
    #     "label": "Doctors",
    #     "tags": {
    #         "amenity": "doctors"
    #     }
    # },

    # "dentists": {
    #     "label": "Dentists",
    #     "tags": {
    #         "amenity": "dentist"
    #     }
    # },

    # "pharmacies": {
    #     "label": "Pharmacies",
    #     "tags": {
    #         "amenity": "pharmacy"
    #     }
    # },

    # "clinics": {
    #     "label": "Clinics",
    #     "tags": {
    #         "amenity": "clinic"
    #     }
    # },

    # "care_homes": {
    #     "label": "Care Homes",
    #     "tags": {
    #         "amenity": "social_facility"
    #     }
    # },

    # "fire_stations": {
    #     "label": "Fire Stations",
    #     "tags": {
    #         "amenity": "fire_station"
    #     }
    # },

    # "police_stations": {
    #     "label": "Police Stations",
    #     "tags": {
    #         "amenity": "police"
    #     }
    # },

    # "community_centres": {
    #     "label": "Community Centres",
    #     "tags": {
    #         "amenity": "community_centre"
    #     }
    # },

    # "places_of_worship": {
    #     "label": "Places of Worship",
    #     "tags": {
    #         "amenity": "place_of_worship"
    #     }
    # },

    # "banks": {
    #     "label": "Banks",
    #     "tags": {
    #         "amenity": "bank"
    #     }
    # },

    # "atms": {
    #     "label": "ATMs",
    #     "tags": {
    #         "amenity": "atm"
    #     }
    # },

    # "post_offices": {
    #     "label": "Post Offices",
    #     "tags": {
    #         "amenity": "post_office"
    #     }
    # },

    # "fuel_stations": {
    #     "label": "Fuel Stations",
    #     "tags": {
    #         "amenity": "fuel"
    #     }
    # },

    # "car_parks": {
    #     "label": "Car Parks",
    #     "tags": {
    #         "amenity": "parking"
    #     }
    # },

    # "taxi_ranks": {
    #     "label": "Taxi Ranks",
    #     "tags": {
    #         "amenity": "taxi"
    #     }
    # },

    # "ferry_terminals": {
    #     "label": "Ferry Terminals",
    #     "tags": {
    #         "amenity": "ferry_terminal"
    #     }
    # },

    # "marinas": {
    #     "label": "Marinas",
    #     "tags": {
    #         "leisure": "marina"
    #     }
    # },

    # "golf_courses": {
    #     "label": "Golf Courses",
    #     "tags": {
    #         "leisure": "golf_course"
    #     }
    # },

    # "swimming_pools": {
    #     "label": "Swimming Pools",
    #     "tags": {
    #         "leisure": "swimming_pool"
    #     }
    # },

    # "playgrounds": {
    #     "label": "Playgrounds",
    #     "tags": {
    #         "leisure": "playground"
    #     }
    # },

    # "nature_reserves": {
    #     "label": "Nature Reserves",
    #     "tags": {
    #         "leisure": "nature_reserve"
    #     }
    # },

    # "camp_sites": {
    #     "label": "Camp Sites",
    #     "tags": {
    #         "tourism": "camp_site"
    #     }
    # },

    # "guest_houses": {
    #     "label": "Guest Houses",
    #     "tags": {
    #         "tourism": "guest_house"
    #     }
    # },

    # "hostels": {
    #     "label": "Hostels",
    #     "tags": {
    #         "tourism": "hostel"
    #     }
    # },

    # "viewpoints": {
    #     "label": "Viewpoints",
    #     "tags": {
    #         "tourism": "viewpoint"
    #     }
    # },

    # "information_centres": {
    #     "label": "Information Centres",
    #     "tags": {
    #         "tourism": "information"
    #     }
    # },

    # "department_stores": {
    #     "label": "Department Stores",
    #     "tags": {
    #         "shop": "department_store"
    #     }
    # },

    # "convenience_stores": {
    #     "label": "Convenience Stores",
    #     "tags": {
    #         "shop": "convenience"
    #     }
    # },

    # "clothes_shops": {
    #     "label": "Clothes Shops",
    #     "tags": {
    #         "shop": "clothes"
    #     }
    # },

    # "bakeries": {
    #     "label": "Bakeries",
    #     "tags": {
    #         "shop": "bakery"
    #     }
    # },

    # "bars": {
    #     "label": "Bars",
    #     "tags": {
    #         "amenity": "bar"
    #     }
    # },

    # "nightclubs": {
    #     "label": "Nightclubs",
    #     "tags": {
    #         "amenity": "nightclub"
    #     }
    # },

    # "food_courts": {
    #     "label": "Food Courts",
    #     "tags": {
    #         "amenity": "food_court"
    #     }
    # },

    # "markets": {
    #     "label": "Markets",
    #     "tags": {
    #         "amenity": "marketplace"
    #     }
    # },

    # "industrial_areas": {
    #     "label": "Industrial Areas",
    #     "tags": {
    #         "landuse": "industrial"
    #     }
    # },

    # "retail_areas": {
    #     "label": "Retail Areas",
    #     "tags": {
    #         "landuse": "retail"
    #     }
    # },

    # "commercial_areas": {
    #     "label": "Commercial Areas",
    #     "tags": {
    #         "landuse": "commercial"
    #     }
    # },

    # "piers": {
    #     "label": "Piers",
    #     "tags": {
    #         "man_made": "pier"
    #     }
    # },

    # "lighthouses": {
    #     "label": "Lighthouses",
    #     "tags": {
    #         "man_made": "lighthouse"
    #     }
    # },

    # "power_stations": {
    #     "label": "Power Stations",
    #     "tags": {
    #         "power": "plant"
    #     }
    # }
}


print("=" * 80)
print("CELL 7 - OPTIONAL POINT OF INTEREST CATEGORY CATALOGUE")
print("=" * 80)


print(
    "\nThe optional category catalogue contains "
    "commented examples only."
)


print(
    "No optional categories are active unless "
    "they are copied into POI_CONFIG in Cell 6."
)

CELL 7 - OPTIONAL POINT OF INTEREST CATEGORY CATALOGUE

The optional category catalogue contains commented examples only.
No optional categories are active unless they are copied into POI_CONFIG in Cell 6.


## What Cell 7 Does

This cell defines additional optional POI categories that are not part of the current active dataset.

They are kept separately from the active `POI_CONFIG` so that they can be reviewed or added later if we want more locations without changing the existing production dataset accidentally.

Only categories currently present in `POI_CONFIG` are downloaded, combined and required by the later validation checks.

# 4. Download Workflow

This section creates the reusable Overpass download functions and processes every POI category currently defined in `POI_CONFIG`.

The workflow downloads each category independently and stores it as its own GeoPackage. Valid existing category files can be reused, allowing interrupted or repeated notebook runs to continue without downloading every category again.

In [9]:
# ================================================================
# CELL 8 - CREATE THE POI DOWNLOAD FUNCTIONS
# ================================================================

download_results = []


def category_output_file(
    poi_type
):
    """
    Return the saved GeoPackage path for one category.
    """

    return (
        POI_DOWNLOAD_FOLDER
        / f"{poi_type}.gpkg"
    )


def inspect_category_download(
    poi_type
):
    """
    Inspect whether a completed category file can be reused.
    """

    output_file = category_output_file(
        poi_type
    )


    result = {
        "Valid": False,
        "Features": 0,
        "CRS": None,
        "Missing Geometries": 0,
        "Empty Geometries": 0,
        "File Size (MB)": 0.0,
        "Reason": None
    }


    if not output_file.exists():

        result[
            "Reason"
        ] = "File does not exist"

        return result


    try:

        saved_gdf = gpd.read_file(
            output_file,
            layer=poi_type
        )


        result[
            "Features"
        ] = len(
            saved_gdf
        )


        result[
            "CRS"
        ] = str(
            saved_gdf.crs
        )


        result[
            "Missing Geometries"
        ] = int(
            saved_gdf.geometry.isna().sum()
        )


        result[
            "Empty Geometries"
        ] = int(
            saved_gdf.geometry.is_empty.sum()
        )


        result[
            "File Size (MB)"
        ] = (
            output_file.stat().st_size
            /
            1_000_000
        )


        positive_features = (
            len(
                saved_gdf
            ) > 0
        )


        correct_crs = bool(
            saved_gdf.crs is not None
            and
            saved_gdf.crs.to_epsg() == 32630
        )


        complete_geometries = bool(
            saved_gdf.geometry.notna().all()
            and
            (
                ~saved_gdf.geometry.is_empty
            ).all()
        )


        positive_file_size = (
            output_file.stat().st_size > 0
        )


        result[
            "Valid"
        ] = bool(
            positive_features
            and
            correct_crs
            and
            complete_geometries
            and
            positive_file_size
        )


        if not positive_features:

            result[
                "Reason"
            ] = "No features"

        elif not correct_crs:

            result[
                "Reason"
            ] = "Incorrect or missing CRS"

        elif not complete_geometries:

            result[
                "Reason"
            ] = "Missing or empty geometries"

        elif not positive_file_size:

            result[
                "Reason"
            ] = "Empty file"

        else:

            result[
                "Reason"
            ] = None


    except Exception as error:

        result[
            "Reason"
        ] = str(
            error
        )


    return result


def valid_category_download(
    poi_type
):
    """
    Return True when one saved category can be reused.
    """

    return inspect_category_download(
        poi_type
    )[
        "Valid"
    ]


def download_one_poi_category(
    poi_type
):
    """
    Download or reuse one configured POI category.
    """

    config = POI_CONFIG[
        poi_type
    ]


    output_file = category_output_file(
        poi_type
    )


    print("-" * 80)

    print(
        f"{config['label']} ({poi_type})"
    )

    print("-" * 80)


    existing_check = inspect_category_download(
        poi_type
    )


    if existing_check[
        "Valid"
    ]:

        result = {
            "POI Type": poi_type,
            "POI Label": config[
                "label"
            ],
            "Status": "Reused valid download",
            "Features": existing_check[
                "Features"
            ],
            "File Size (MB)": existing_check[
                "File Size (MB)"
            ],
            "Processing Time (Seconds)": 0.0,
            "File": str(
                output_file
            ),
            "Error": None
        }


        download_results.append(
            result
        )


        print(
            f"Reused valid download: "
            f"{existing_check['Features']:,} features"
        )


        return result


    if output_file.exists():

        output_file.unlink()


    request_start_time = time.time()


    try:

        print(
            "Sending one request for this category "
            "to the main Overpass server..."
        )


        downloaded_gdf = ox.features_from_bbox(
            OVERPASS_BBOX,
            tags=config[
                "tags"
            ]
        )


        if downloaded_gdf.empty:

            raise ValueError(
                "The Overpass request returned no features."
            )


        downloaded_gdf = downloaded_gdf.to_crs(
            PROJECTED_CRS
        )


        downloaded_gdf = gpd.clip(
            downloaded_gdf,
            buffered_download_area.geometry.iloc[0]
        )


        downloaded_gdf = (
            downloaded_gdf.loc[
                downloaded_gdf.geometry.notna()
                &
                ~downloaded_gdf.geometry.is_empty
            ]
            .copy()
        )


        if downloaded_gdf.empty:

            raise ValueError(
                "No valid features remained inside "
                "the 5 km buffered download area."
            )


        temporary_output_file = (
            output_file.with_name(
                output_file.stem
                +
                ".temporary.gpkg"
            )
        )


        if temporary_output_file.exists():

            temporary_output_file.unlink()


        downloaded_gdf.to_file(
            temporary_output_file,
            layer=poi_type,
            driver="GPKG"
        )


        assert temporary_output_file.exists()

        assert temporary_output_file.stat().st_size > 0


        if output_file.exists():

            output_file.unlink()


        temporary_output_file.replace(
            output_file
        )


        completed_check = inspect_category_download(
            poi_type
        )


        if not completed_check[
            "Valid"
        ]:

            raise ValueError(
                "The saved category file did not pass "
                "the validation checks."
            )


        elapsed_seconds = (
            time.time()
            -
            request_start_time
        )


        result = {
            "POI Type": poi_type,
            "POI Label": config[
                "label"
            ],
            "Status": "Downloaded",
            "Features": completed_check[
                "Features"
            ],
            "File Size (MB)": completed_check[
                "File Size (MB)"
            ],
            "Processing Time (Seconds)": round(
                elapsed_seconds,
                2
            ),
            "File": str(
                output_file
            ),
            "Error": None
        }


        download_results.append(
            result
        )


        print(
            f"Downloaded and saved "
            f"{completed_check['Features']:,} features "
            f"in {elapsed_seconds:,.2f} seconds."
        )


        return result


    except Exception as error:

        elapsed_seconds = (
            time.time()
            -
            request_start_time
        )


        temporary_output_file = (
            output_file.with_name(
                output_file.stem
                +
                ".temporary.gpkg"
            )
        )


        if temporary_output_file.exists():

            temporary_output_file.unlink()


        result = {
            "POI Type": poi_type,
            "POI Label": config[
                "label"
            ],
            "Status": "Failed",
            "Features": 0,
            "File Size (MB)": 0.0,
            "Processing Time (Seconds)": round(
                elapsed_seconds,
                2
            ),
            "File": str(
                output_file
            ),
            "Error": str(
                error
            )
        }


        download_results.append(
            result
        )


        print(
            f"Download failed after "
            f"{elapsed_seconds:,.2f} seconds."
        )


        print(
            str(
                error
            )
        )


        return result


def download_all_configured_categories():
    """
    Process every category currently defined in POI_CONFIG.
    """

    current_run_results = []


    for category_number, poi_type in enumerate(
        CONFIGURED_POI_TYPES,
        start=1
    ):

        print(
            f"\nCATEGORY {category_number}/"
            f"{CONFIGURED_POI_CATEGORY_COUNT}"
        )


        current_run_results.append(
            download_one_poi_category(
                poi_type
            )
        )


    return pd.DataFrame(
        current_run_results
    )


print("=" * 80)
print("CELL 8 - CREATE THE POI DOWNLOAD FUNCTIONS")
print("=" * 80)


print(
    "\nThe reusable POI download and validation "
    "functions were created successfully."
)

CELL 8 - CREATE THE POI DOWNLOAD FUNCTIONS

The reusable POI download and validation functions were created successfully.


## What Cell 8 Does

This cell defines the functions used by the Overpass download workflow.

The functions:

- construct the Overpass query for each configured category;
- request matching OpenStreetMap nodes, ways and relations;
- retry requests when necessary;
- convert returned features to representative point locations;
- transform those locations to EPSG:32630;
- standardise the downloaded attributes;
- save each completed category safely to its own GeoPackage;
- validate existing files before allowing them to be reused.

These functions provide the reusable download and processing logic used by the next cell.

In [10]:
# ================================================================
# CELL 9 - DOWNLOAD ALL CONFIGURED POI CATEGORIES
# ================================================================

all_downloads_start_time = time.time()

POI_MAX_RETRY_ATTEMPTS = 2

POI_RETRY_WAIT_SECONDS = (
    180
)

print("=" * 80)
print("CELL 9 - DOWNLOAD ALL CONFIGURED POI CATEGORIES")
print("=" * 80)

print(
    f"\nConfigured categories: "
    f"{CONFIGURED_POI_CATEGORY_COUNT:,}"
)

print(
    f"\nMaximum download attempts for a missing "
    f"or invalid category: "
    f"{POI_MAX_RETRY_ATTEMPTS}"
)

print(
    f"Wait after a failed attempt: "
    f"{POI_RETRY_WAIT_SECONDS / 60:,.1f} minutes"
)

print(
    "\nValid saved category files will be reused "
    "and will not be downloaded again."
)

# ------------------------------------------------
# Process every configured category
# ------------------------------------------------

completed_category_results = []

for category_number, poi_type in enumerate(
    CONFIGURED_POI_TYPES,
    start=1
):

    config = POI_CONFIG[
        poi_type
    ]


    print(
        "\n"
        +
        "=" * 80
    )


    print(
        f"CATEGORY {category_number}/"
        f"{CONFIGURED_POI_CATEGORY_COUNT}"
    )


    print(
        f"{config['label']} "
        f"({poi_type})"
    )


    print(
        "=" * 80
    )


    # ------------------------------------------------
    # Check first whether this category is already
    # complete
    # ------------------------------------------------

    existing_check = (
        inspect_category_download(
            poi_type
        )
    )


    if existing_check[
        "Valid"
    ]:

        result = (
            download_one_poi_category(
                poi_type
            )
        )


        result[
            "Attempts This Run"
        ] = 0


        completed_category_results.append(
            result
        )


        print(
            "Category already complete. "
            "Moving to the next category."
        )


        continue


    # ------------------------------------------------
    # Missing/invalid category:
    # try a maximum of two times
    # ------------------------------------------------

    category_completed = False


    for attempt_number in range(
        1,
        POI_MAX_RETRY_ATTEMPTS + 1
    ):

        print(
            "\n"
            +
            "-" * 80
        )


        print(
            f"Attempt {attempt_number:,}/"
            f"{POI_MAX_RETRY_ATTEMPTS:,} "
            f"for {config['label']}"
        )


        print(
            "-" * 80
        )


        result = (
            download_one_poi_category(
                poi_type
            )
        )


        # ------------------------------------------------
        # Success
        # ------------------------------------------------

        if result[
            "Status"
        ] in [
            "Downloaded",
            "Reused valid download"
        ]:

            result[
                "Attempts This Run"
            ] = attempt_number


            completed_category_results.append(
                result
            )


            print(
                f"\nSUCCESS: "
                f"{config['label']} completed "
                f"after {attempt_number:,} attempt(s)."
            )


            print(
                "Moving to the next category."
            )


            category_completed = True


            break


        # ------------------------------------------------
        # Failure
        # ------------------------------------------------

        print(
            f"\nAttempt {attempt_number:,} failed."
        )


        print(
            "Error:"
        )


        print(
            result[
                "Error"
            ]
        )


        if (
            attempt_number
            <
            POI_MAX_RETRY_ATTEMPTS
        ):

            print(
                f"\nWaiting "
                f"{POI_RETRY_WAIT_SECONDS / 60:,.1f} "
                f"minutes before trying "
                f"{config['label']} again..."
            )


            time.sleep(
                POI_RETRY_WAIT_SECONDS
            )


    # ------------------------------------------------
    # Category still failed after maximum attempts
    # ------------------------------------------------

    if not category_completed:

        result[
            "Attempts This Run"
        ] = POI_MAX_RETRY_ATTEMPTS


        completed_category_results.append(
            result
        )


        print(
            f"\nFAILED: "
            f"{config['label']} did not complete "
            f"after {POI_MAX_RETRY_ATTEMPTS:,} attempts."
        )


        print(
            "Moving to the next category."
        )


# ------------------------------------------------
# Final summary
# ------------------------------------------------

all_poi_download_summary = pd.DataFrame(
    completed_category_results
)

all_downloads_seconds = (
    time.time()
    -
    all_downloads_start_time
)

downloaded_category_count = int(
    (
        all_poi_download_summary[
            "Status"
        ]
        ==
        "Downloaded"
    ).sum()
)

reused_category_count = int(
    (
        all_poi_download_summary[
            "Status"
        ]
        ==
        "Reused valid download"
    ).sum()
)

failed_category_count = int(
    (
        all_poi_download_summary[
            "Status"
        ]
        ==
        "Failed"
    ).sum()
)

successful_or_reused_count = (
    downloaded_category_count
    +
    reused_category_count
)

complete_download_summary = pd.DataFrame(
    {
        "Measure": [
            "Configured categories",
            "Newly downloaded categories",
            "Valid categories reused",
            "Successful or reused categories",
            "Failed categories",
            "Available features",
            "Total retry attempts this run",
            "Total processing time (minutes)"
        ],
        "Value": [
            CONFIGURED_POI_CATEGORY_COUNT,
            downloaded_category_count,
            reused_category_count,
            successful_or_reused_count,
            failed_category_count,
            int(
                all_poi_download_summary[
                    "Features"
                ].sum()
            ),
            int(
                all_poi_download_summary[
                    "Attempts This Run"
                ].sum()
            ),
            round(
                all_downloads_seconds / 60,
                2
            )
        ]
    }
)

print(
    "\n"
    +
    "=" * 80
)

print(
    "ALL CONFIGURED POI CATEGORIES PROCESSED"
)

print(
    "=" * 80
)

print(
    "\nCategory results:"
)

display(
    all_poi_download_summary[
        [
            "POI Type",
            "POI Label",
            "Status",
            "Attempts This Run",
            "Features",
            "File Size (MB)",
            "Processing Time (Seconds)",
            "Error"
        ]
    ]
)

print(
    "\nComplete download summary:"
)

display(
    complete_download_summary
)

# ------------------------------------------------
# Final validation
# ------------------------------------------------

assert len(
    all_poi_download_summary
) == CONFIGURED_POI_CATEGORY_COUNT

print(
    f"\nSuccessful or reused categories: "
    f"{successful_or_reused_count:,}/"
    f"{CONFIGURED_POI_CATEGORY_COUNT:,}"
)

print(
    f"Failed categories: "
    f"{failed_category_count:,}"
)

if failed_category_count == 0:

    for poi_type in CONFIGURED_POI_TYPES:

        assert inspect_category_download(
            poi_type
        )[
            "Valid"
        ]


    print(
        "\nEvery configured POI category now has "
        "a valid saved download."
    )


    print(
        "\nThe configured category download process "
        "has finished successfully."
    )

else:

    print(
        "\nSome configured POI categories could not "
        "be downloaded during this run."
    )


    print(
        "The successful categories were kept and "
        "the failed categories can be retried by "
        "running this cell again."
    )

CELL 9 - DOWNLOAD ALL CONFIGURED POI CATEGORIES

Configured categories: 24

Maximum download attempts for a missing or invalid category: 2
Wait after a failed attempt: 3.0 minutes

Valid saved category files will be reused and will not be downloaded again.

CATEGORY 1/24
Stadiums (stadiums)
--------------------------------------------------------------------------------
Stadiums (stadiums)
--------------------------------------------------------------------------------
Reused valid download: 11 features
Category already complete. Moving to the next category.

CATEGORY 2/24
Sport Centres (sport_centres)
--------------------------------------------------------------------------------
Sport Centres (sport_centres)
--------------------------------------------------------------------------------
Reused valid download: 255 features
Category already complete. Moving to the next category.

CATEGORY 3/24
Holiday Parks (holiday_parks)
-------------------------------------------------------------

,POI Type,POI Label,Status,Attempts This Run,Features,File Size (MB),Processing Time (Seconds),Error
0,stadiums,Stadiums,Reused valid download,0,11,0.106496,0.0,None
1,sport_centres,Sport Centres,Reused valid download,0,255,0.241664,0.0,None
2,holiday_parks,Holiday Parks,Reused valid download,0,195,0.221184,0.0,None
3,theme_parks,Theme Parks,Reused valid download,0,21,0.110592,0.0,None
4,zoos,Zoos,Reused valid download,0,17,0.110592,0.0,None
5,railway_stations,Railway Stations,Reused valid download,0,112,0.139264,0.0,None
6,bus_stations,Bus Stations,Reused valid download,0,23,0.106496,0.0,None
7,bus_stops,Bus Stops,Reused valid download,0,11117,3.952640,0.0,None
8,airports,Airports,Reused valid download,0,27,0.126976,0.0,None
9,hospitals,Hospitals,Reused valid download,0,79,0.155648,0.0,None



Complete download summary:


,Measure,Value
0,Configured categories,24.00
1,Newly downloaded categories,0.00
2,Valid categories reused,24.00
3,Successful or reused categories,24.00
4,Failed categories,0.00
5,Available features,23987.00
6,Total retry attempts this run,0.00
7,Total processing time (minutes),0.04



Successful or reused categories: 24/24
Failed categories: 0

Every configured POI category now has a valid saved download.

The configured category download process has finished successfully.


## What Cell 9 Does

This cell processes every POI category currently defined in `POI_CONFIG`.

For each category, the workflow first checks whether a valid completed GeoPackage already exists.

If it does, that file is reused.

If it does not, the category is downloaded from Overpass, processed and saved.

A processing summary records the result for every configured category.

This makes the download stage restartable and avoids repeating successful Overpass requests unnecessarily.

# 5. Download Verification

This section verifies the complete set of individual POI category downloads before they are allowed to enter the combined dataset.

The purpose is to ensure that the downstream dataset is created only from valid and complete category files.

In [11]:
# ================================================================
# CELL 10 - VERIFY ALL CONFIGURED CATEGORY DOWNLOADS
# ================================================================

category_inventory_records = []


for category_number, (
    poi_type,
    config
) in enumerate(
    POI_CONFIG.items(),
    start=1
):

    output_file = category_output_file(
        poi_type
    )


    category_check = inspect_category_download(
        poi_type
    )


    if category_check[
        "Valid"
    ]:

        category_gdf = gpd.read_file(
            output_file,
            layer=poi_type
        )


        geometry_types = ", ".join(
            sorted(
                category_gdf.geometry.geom_type.unique()
            )
        )


    else:

        geometry_types = None


    category_inventory_records.append(
        {
            "Category Number": category_number,
            "POI Type": poi_type,
            "POI Label": config[
                "label"
            ],
            "File Exists": output_file.exists(),
            "Valid Download": category_check[
                "Valid"
            ],
            "Features": category_check[
                "Features"
            ],
            "CRS": category_check[
                "CRS"
            ],
            "Geometry Types": geometry_types,
            "Missing Geometries": category_check[
                "Missing Geometries"
            ],
            "Empty Geometries": category_check[
                "Empty Geometries"
            ],
            "File Size (MB)": category_check[
                "File Size (MB)"
            ],
            "Validation Reason": category_check[
                "Reason"
            ],
            "File": str(
                output_file
            )
        }
    )


poi_download_inventory = pd.DataFrame(
    category_inventory_records
)


valid_category_count = int(
    poi_download_inventory[
        "Valid Download"
    ].sum()
)


invalid_category_count = int(
    (
        ~poi_download_inventory[
            "Valid Download"
        ]
    ).sum()
)


download_verification_summary = pd.DataFrame(
    {
        "Measure": [
            "Configured categories",
            "Valid category files",
            "Missing or invalid category files",
            "Total available features",
            "Combined category-file size (MB)"
        ],
        "Value": [
            CONFIGURED_POI_CATEGORY_COUNT,
            valid_category_count,
            invalid_category_count,
            int(
                poi_download_inventory[
                    "Features"
                ].sum()
            ),
            float(
                poi_download_inventory[
                    "File Size (MB)"
                ].sum()
            )
        ]
    }
)


incomplete_categories = (
    poi_download_inventory.loc[
        ~poi_download_inventory[
            "Valid Download"
        ],
        "POI Type"
    ]
    .tolist()
)


print("=" * 80)
print("CELL 10 - VERIFY ALL CONFIGURED CATEGORY DOWNLOADS")
print("=" * 80)


display(
    poi_download_inventory
)


print(
    "\nDownload verification summary:"
)


display(
    download_verification_summary
)


if incomplete_categories:

    print(
        "\nThe following configured categories must "
        "be completed before the final dataset can "
        "be created:"
    )


    for poi_type in incomplete_categories:

        print(
            f"- {poi_type}"
        )


assert len(
    poi_download_inventory
) == CONFIGURED_POI_CATEGORY_COUNT


assert not incomplete_categories, (
    "Not every configured POI category currently "
    "has a valid saved file. Rerun Cell 9 to "
    "attempt the missing categories again."
)


assert valid_category_count == (
    CONFIGURED_POI_CATEGORY_COUNT
)


print(
    "\nEvery configured POI category file was "
    "verified successfully."
)

CELL 10 - VERIFY ALL CONFIGURED CATEGORY DOWNLOADS


,Category Number,POI Type,POI Label,File Exists,Valid Download,Features,CRS,Geometry Types,Missing Geometries,Empty Geometries,File Size (MB),Validation Reason,File
0,1,stadiums,Stadiums,True,True,11,EPSG:32630,"Point, Polygon",0,0,0.106496,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
1,2,sport_centres,Sport Centres,True,True,255,EPSG:32630,"MultiPolygon, Point, Polygon",0,0,0.241664,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
2,3,holiday_parks,Holiday Parks,True,True,195,EPSG:32630,"Point, Polygon",0,0,0.221184,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
3,4,theme_parks,Theme Parks,True,True,21,EPSG:32630,"Point, Polygon",0,0,0.110592,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
4,5,zoos,Zoos,True,True,17,EPSG:32630,"Point, Polygon",0,0,0.110592,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
5,6,railway_stations,Railway Stations,True,True,112,EPSG:32630,Point,0,0,0.139264,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
6,7,bus_stations,Bus Stations,True,True,23,EPSG:32630,"Point, Polygon",0,0,0.106496,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
7,8,bus_stops,Bus Stops,True,True,11117,EPSG:32630,"Point, Polygon",0,0,3.952640,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
8,9,airports,Airports,True,True,27,EPSG:32630,"Point, Polygon",0,0,0.126976,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
9,10,hospitals,Hospitals,True,True,79,EPSG:32630,"Point, Polygon",0,0,0.155648,None,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...



Download verification summary:


,Measure,Value
0,Configured categories,24.00000
1,Valid category files,24.00000
2,Missing or invalid category files,0.00000
3,Total available features,23987.00000
4,Combined category-file size (MB),13.94688



Every configured POI category file was verified successfully.


## What Cell 10 Does

This cell independently checks every POI category currently configured in `POI_CONFIG`.

For each category it verifies that:

- the expected GeoPackage exists;
- the expected layer can be read;
- the file contains observations;
- the required fields are present;
- the geometry is available;
- the projected coordinate reference system is correct.

The notebook continues to dataset preparation only when every configured category has a valid completed download.

# 6. POI Dataset Preparation

This section loads the verified category GeoPackages, standardises their fields and combines them into one validated POI dataset.

At this stage the internal category field remains `poi_type`. The final field name required by `2_Cleaning_and_Feature_Engineering` (Notebook 2) is applied in the following compatibility section.

In [12]:
# ================================================================
# CELL 11 - STANDARDISE ALL CONFIGURED POI CATEGORIES
# ================================================================

standardised_poi_frames = []

standardisation_records = []


def geometry_to_location_point(
    geometry
):
    """
    Convert an OpenStreetMap geometry into one location point.
    """

    if geometry is None:

        return None


    if geometry.is_empty:

        return None


    if geometry.geom_type == "Point":

        return geometry


    if geometry.geom_type == "MultiPoint":

        return geometry.centroid


    return geometry.representative_point()


for category_number, (
    poi_type,
    config
) in enumerate(
    POI_CONFIG.items(),
    start=1
):

    category_file = category_output_file(
        poi_type
    )


    category_gdf = gpd.read_file(
        category_file,
        layer=poi_type
    )


    original_category_rows = len(
        category_gdf
    )


    if category_gdf.crs is None:

        raise ValueError(
            f"{poi_type} does not contain a CRS."
        )


    if category_gdf.crs.to_epsg() != 32630:

        category_gdf = category_gdf.to_crs(
            PROJECTED_CRS
        )


    category_gdf = (
        category_gdf.loc[
            category_gdf.geometry.notna()
            &
            ~category_gdf.geometry.is_empty
        ]
        .copy()
    )


    category_gdf = gpd.clip(
        category_gdf,
        buffered_download_area.geometry.iloc[0]
    )


    category_gdf = (
        category_gdf.loc[
            category_gdf.geometry.notna()
            &
            ~category_gdf.geometry.is_empty
        ]
        .copy()
    )


    category_gdf = (
        category_gdf.reset_index()
    )


    if "element" in category_gdf.columns:

        element_column = (
            "element"
        )


    elif "element_type" in category_gdf.columns:

        element_column = (
            "element_type"
        )


    else:

        category_gdf[
            "element"
        ] = "unknown"

        element_column = (
            "element"
        )


    if "id" in category_gdf.columns:

        osm_id_column = (
            "id"
        )


    elif "osmid" in category_gdf.columns:

        osm_id_column = (
            "osmid"
        )


    else:

        category_gdf[
            "id"
        ] = np.arange(
            1,
            len(
                category_gdf
            ) + 1,
            dtype="int64"
        )

        osm_id_column = (
            "id"
        )


    if "name" not in category_gdf.columns:

        category_gdf[
            "name"
        ] = pd.NA


    category_points = (
        category_gdf.geometry.apply(
            geometry_to_location_point
        )
    )


    standardised_category = gpd.GeoDataFrame(
        {
            "poi_type": pd.Series(
                poi_type,
                index=category_gdf.index,
                dtype="string"
            ),

            "poi_name": (
                category_gdf[
                    "name"
                ].astype(
                    "string"
                )
            ),

            "osm_element_type": (
                category_gdf[
                    element_column
                ].astype(
                    "string"
                )
            ),

            "osm_id": (
                pd.to_numeric(
                    category_gdf[
                        osm_id_column
                    ],
                    errors="coerce"
                ).astype(
                    "Int64"
                )
            )
        },
        geometry=category_points,
        crs=PROJECTED_CRS
    )


    standardised_category = (
        standardised_category.loc[
            standardised_category.geometry.notna()
            &
            ~standardised_category.geometry.is_empty
        ]
        .copy()
    )


    standardised_category[
        "poi_name"
    ] = (
        standardised_category[
            "poi_name"
        ]
        .fillna(
            config[
                "label"
            ]
        )
        .replace(
            {
                "<NA>": config[
                    "label"
                ],
                "nan": config[
                    "label"
                ],
                "None": config[
                    "label"
                ],
                "": config[
                    "label"
                ]
            }
        )
    )


    standardised_category[
        "poi_source"
    ] = (
        "OpenStreetMap Overpass"
    )


    standardised_category[
        "download_buffer_metres"
    ] = (
        POI_DOWNLOAD_BUFFER_METRES
    )


    standardised_poi_frames.append(
        standardised_category
    )


    standardisation_records.append(
        {
            "Category Number": category_number,
            "POI Type": poi_type,
            "POI Label": config[
                "label"
            ],
            "Downloaded Features": (
                original_category_rows
            ),
            "Features After Buffer Check": len(
                category_gdf
            ),
            "Standardised Features": len(
                standardised_category
            ),
            "Features With OSM Names": int(
                (
                    standardised_category[
                        "poi_name"
                    ]
                    !=
                    config[
                        "label"
                    ]
                ).sum()
            ),
            "Fallback Category Names": int(
                (
                    standardised_category[
                        "poi_name"
                    ]
                    ==
                    config[
                        "label"
                    ]
                ).sum()
            )
        }
    )


poi_standardisation_summary = pd.DataFrame(
    standardisation_records
)


print("=" * 80)
print("CELL 11 - STANDARDISE ALL CONFIGURED POI CATEGORIES")
print("=" * 80)


display(
    poi_standardisation_summary
)


assert len(
    standardised_poi_frames
) == CONFIGURED_POI_CATEGORY_COUNT


assert len(
    poi_standardisation_summary
) == CONFIGURED_POI_CATEGORY_COUNT


assert (
    poi_standardisation_summary[
        "Standardised Features"
    ] > 0
).all()


print(
    "\nEvery configured POI category was "
    "standardised successfully."
)

CELL 11 - STANDARDISE ALL CONFIGURED POI CATEGORIES


,Category Number,POI Type,POI Label,Downloaded Features,Features After Buffer Check,Standardised Features,Features With OSM Names,Fallback Category Names
0,1,stadiums,Stadiums,11,11,11,10,1
1,2,sport_centres,Sport Centres,255,255,255,219,36
2,3,holiday_parks,Holiday Parks,195,195,195,150,45
3,4,theme_parks,Theme Parks,21,21,21,20,1
4,5,zoos,Zoos,17,17,17,17,0
5,6,railway_stations,Railway Stations,112,112,112,111,1
6,7,bus_stations,Bus Stations,23,23,23,21,2
7,8,bus_stops,Bus Stops,11117,11117,11117,10997,120
8,9,airports,Airports,27,27,27,27,0
9,10,hospitals,Hospitals,79,79,79,77,2



Every configured POI category was standardised successfully.


## What Cell 11 Does

This cell loads every verified POI category GeoPackage and converts it to a common structure.

The standardised records retain:

- the POI category;
- the POI name;
- the OpenStreetMap element type;
- the OpenStreetMap identifier;
- the data source;
- the POI download buffer distance;
- the projected point geometry.

Missing POI names are replaced with a consistent fallback value so that the final dataset contains a usable name for every observation.

The resulting category datasets are then ready to be combined.

In [13]:
# ================================================================
# CELL 12 - COMBINE AND VALIDATE THE POI DATASET
# ================================================================

combined_poi_dataset = gpd.GeoDataFrame(
    pd.concat(
        standardised_poi_frames,
        ignore_index=True
    ),
    geometry="geometry",
    crs=PROJECTED_CRS
)


combined_poi_dataset[
    "poi_id"
] = (
    combined_poi_dataset[
        "poi_type"
    ].astype(
        "string"
    )
    +
    "_"
    +
    combined_poi_dataset[
        "osm_element_type"
    ].astype(
        "string"
    )
    +
    "_"
    +
    combined_poi_dataset[
        "osm_id"
    ].astype(
        "string"
    )
)


rows_before_duplicate_removal = len(
    combined_poi_dataset
)


combined_poi_dataset = (
    combined_poi_dataset[
        [
            "poi_id",
            "poi_type",
            "poi_name",
            "osm_element_type",
            "osm_id",
            "poi_source",
            "download_buffer_metres",
            "geometry"
        ]
    ]
    .drop_duplicates(
        subset=[
            "poi_id"
        ],
        keep="first"
    )
    .sort_values(
        [
            "poi_type",
            "poi_name",
            "poi_id"
        ],
        kind="mergesort"
    )
    .reset_index(
        drop=True
    )
)


duplicate_rows_removed = (
    rows_before_duplicate_removal
    -
    len(
        combined_poi_dataset
    )
)


combined_category_summary = (
    combined_poi_dataset.groupby(
        "poi_type",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "Features"
        }
    )
)


configured_categories_missing_from_combined = sorted(
    set(
        CONFIGURED_POI_TYPES
    )
    -
    set(
        combined_poi_dataset[
            "poi_type"
        ].unique()
    )
)


unexpected_categories_in_combined = sorted(
    set(
        combined_poi_dataset[
            "poi_type"
        ].unique()
    )
    -
    set(
        CONFIGURED_POI_TYPES
    )
)


combined_validation_summary = pd.DataFrame(
    {
        "Check": [
            "Configured POI categories",
            "Observed POI categories",
            "Combined POI features",
            "Rows removed as duplicate POI identifiers",
            "Unique POI identifiers",
            "Missing configured categories",
            "Unexpected categories",
            "Missing POI identifiers",
            "Missing POI types",
            "Missing POI names",
            "Missing geometries",
            "Empty geometries",
            "Non-point geometries",
            "Duplicate POI identifiers",
            "CRS"
        ],
        "Result": [
            CONFIGURED_POI_CATEGORY_COUNT,
            combined_poi_dataset[
                "poi_type"
            ].nunique(),
            len(
                combined_poi_dataset
            ),
            duplicate_rows_removed,
            combined_poi_dataset[
                "poi_id"
            ].nunique(),
            len(
                configured_categories_missing_from_combined
            ),
            len(
                unexpected_categories_in_combined
            ),
            int(
                combined_poi_dataset[
                    "poi_id"
                ].isna().sum()
            ),
            int(
                combined_poi_dataset[
                    "poi_type"
                ].isna().sum()
            ),
            int(
                combined_poi_dataset[
                    "poi_name"
                ].isna().sum()
            ),
            int(
                combined_poi_dataset.geometry.isna().sum()
            ),
            int(
                combined_poi_dataset.geometry.is_empty.sum()
            ),
            int(
                (
                    combined_poi_dataset.geometry.geom_type
                    !=
                    "Point"
                ).sum()
            ),
            int(
                combined_poi_dataset[
                    "poi_id"
                ].duplicated().sum()
            ),
            str(
                combined_poi_dataset.crs
            )
        ]
    }
)


print("=" * 80)
print("CELL 12 - COMBINE AND VALIDATE THE POI DATASET")
print("=" * 80)


display(
    combined_validation_summary
)


print(
    "\nFeatures by configured POI category:"
)


display(
    combined_category_summary
)


if configured_categories_missing_from_combined:

    print(
        "\nConfigured categories missing from the "
        "combined dataset:"
    )

    for poi_type in (
        configured_categories_missing_from_combined
    ):

        print(
            f"- {poi_type}"
        )


if unexpected_categories_in_combined:

    print(
        "\nUnexpected categories found in the "
        "combined dataset:"
    )

    for poi_type in (
        unexpected_categories_in_combined
    ):

        print(
            f"- {poi_type}"
        )


assert len(
    combined_poi_dataset
) > 0


assert combined_poi_dataset[
    "poi_type"
].nunique() == CONFIGURED_POI_CATEGORY_COUNT


assert not configured_categories_missing_from_combined


assert not unexpected_categories_in_combined


assert combined_poi_dataset[
    "poi_id"
].notna().all()


assert combined_poi_dataset[
    "poi_id"
].is_unique


assert combined_poi_dataset[
    "poi_type"
].notna().all()


assert combined_poi_dataset[
    "poi_name"
].notna().all()


assert combined_poi_dataset.geometry.notna().all()


assert (
    ~combined_poi_dataset.geometry.is_empty
).all()


assert (
    combined_poi_dataset.geometry.geom_type
    ==
    "Point"
).all()


assert combined_poi_dataset.crs.to_epsg() == 32630


print(
    "\nThe combined POI dataset passed all "
    "validation checks."
)

CELL 12 - COMBINE AND VALIDATE THE POI DATASET


,Check,Result
0,Configured POI categories,24
1,Observed POI categories,24
2,Combined POI features,23948
3,Rows removed as duplicate POI identifiers,39
4,Unique POI identifiers,23948
5,Missing configured categories,0
6,Unexpected categories,0
7,Missing POI identifiers,0
8,Missing POI types,0
9,Missing POI names,0



Features by configured POI category:


,poi_type,Features
0,airports,27
1,attractions,302
2,beaches,481
3,bus_stations,23
4,bus_stops,11117
5,cafes,1704
6,fast_foods,1632
7,gyms,136
8,holiday_parks,195
9,hospitals,78



The combined POI dataset passed all validation checks.


## What Cell 12 Does

This cell combines all standardised POI categories into one dataset.

A unique `poi_id` is created from:

- the POI category;
- the OpenStreetMap element type;
- the OpenStreetMap identifier.

Duplicate POI identifiers are removed.

The combined dataset is then validated to confirm that:

- every currently configured category is represented;
- no unexpected category is present;
- every identifier is complete and unique;
- every POI has a usable name;
- every geometry is a valid point;
- the combined dataset uses EPSG:32630.

The expected category count is calculated dynamically from `POI_CONFIG`.

At this stage the internal category field is still named `poi_type`.

The next section converts this to the `poi_category` field required by `2_Cleaning_and_Feature_Engineering` (Notebook 2).

# 7. `2_Cleaning_and_Feature_Engineering` (Notebook 2) Compatibility

This section creates the formal handoff between `1_POI_Dataset_Overpass` (Notebook 1) and `2_Cleaning_and_Feature_Engineering` (Notebook 2).

The validated Overpass dataset is converted to the exact POI schema expected by `2_Cleaning_and_Feature_Engineering` (Notebook 2) without changing the POI observations, categories or spatial locations.

The resulting dataset is then checked before it is written to disk.

In [14]:
# ================================================================
# CELL 13 - PREPARE THE NOTEBOOK 2 POI SCHEMA
# ================================================================

notebook_2_poi_dataset = (
    combined_poi_dataset
    .copy()
)


# ------------------------------------------------
# Rename the POI category field for Notebook 2
# ------------------------------------------------

notebook_2_poi_dataset = (
    notebook_2_poi_dataset
    .rename(
        columns={
            "poi_type": "poi_category"
        }
    )
)


# ------------------------------------------------
# Define the final column order
# ------------------------------------------------

NOTEBOOK_2_POI_COLUMNS = [
    "poi_id",
    "poi_category",
    "poi_name",
    "osm_element_type",
    "osm_id",
    "poi_source",
    "download_buffer_metres",
    "geometry"
]


notebook_2_poi_dataset = (
    notebook_2_poi_dataset[
        NOTEBOOK_2_POI_COLUMNS
    ]
)


# ------------------------------------------------
# Preserve the GeoDataFrame structure
# ------------------------------------------------

notebook_2_poi_dataset = gpd.GeoDataFrame(
    notebook_2_poi_dataset,
    geometry="geometry",
    crs=PROJECTED_CRS
)


print("=" * 80)
print("CELL 13 - PREPARE THE NOTEBOOK 2 POI SCHEMA")
print("=" * 80)


print(
    f"\nRows prepared       : "
    f"{len(notebook_2_poi_dataset):,}"
)

print(
    f"Columns prepared    : "
    f"{len(notebook_2_poi_dataset.columns):,}"
)

print(
    f"POI categories      : "
    f"{notebook_2_poi_dataset['poi_category'].nunique():,}"
)

print(
    f"CRS                 : "
    f"{notebook_2_poi_dataset.crs}"
)


print("\nFinal columns:")

for column_name in notebook_2_poi_dataset.columns:

    print(
        f"  - {column_name}"
    )


display(
    notebook_2_poi_dataset.head()
)

CELL 13 - PREPARE THE NOTEBOOK 2 POI SCHEMA

Rows prepared       : 23,948
Columns prepared    : 8
POI categories      : 24
CRS                 : EPSG:32630

Final columns:
  - poi_id
  - poi_category
  - poi_name
  - osm_element_type
  - osm_id
  - poi_source
  - download_buffer_metres
  - geometry


,poi_id,poi_category,poi_name,osm_element_type,osm_id,poi_source,download_buffer_metres,geometry
0,airports_node_6426124489,airports,Bagber Farm Airstrip,node,6426124489,OpenStreetMap Overpass,5000,POINT (551748.041 5627485.477)
1,airports_relation_6071601,airports,Bembridge Airport,relation,6071601,OpenStreetMap Overpass,5000,POINT (633547.451 5615846.304)
2,airports_node_7392010732,airports,Bere Farm Airstrip,node,7392010732,OpenStreetMap Overpass,5000,POINT (629633.813 5638639.345)
3,airports_node_6183617446,airports,Binstead Airfield,node,6183617446,OpenStreetMap Overpass,5000,POINT (626729.196 5620571.32)
4,airports_node_4583931615,airports,Bognor Regis,node,4583931615,OpenStreetMap Overpass,5000,POINT (664988.089 5630422.953)


## What Cell 13 Does

This cell prepares the completed POI dataset for `2_Cleaning_and_Feature_Engineering` (Notebook 2).

A copy of the validated combined dataset is created so that the original intermediate dataset remains unchanged.

The internal field:

`poi_type`

is renamed to:

`poi_category`

The final column order is defined as:

- `poi_id`
- `poi_category`
- `poi_name`
- `osm_element_type`
- `osm_id`
- `poi_source`
- `download_buffer_metres`
- `geometry`

The dataset remains a GeoDataFrame in EPSG:32630.

This step changes the output schema only. It does not change the POI observations, category assignments or coordinates.

In [15]:
# ================================================================
# CELL 14 - VERIFY NOTEBOOK 2 COMPATIBILITY
# ================================================================

NOTEBOOK_2_REQUIRED_COLUMNS = [
    "poi_category",
    "poi_name",
    "geometry"
]


missing_notebook_2_columns = [
    column_name
    for column_name
    in NOTEBOOK_2_REQUIRED_COLUMNS
    if column_name
    not in notebook_2_poi_dataset.columns
]


missing_category_values = int(
    notebook_2_poi_dataset[
        "poi_category"
    ]
    .isna()
    .sum()
)


missing_geometries = int(
    notebook_2_poi_dataset
    .geometry
    .isna()
    .sum()
)


empty_geometries = int(
    notebook_2_poi_dataset
    .geometry
    .is_empty
    .sum()
)


invalid_geometries = int(
    (
        ~notebook_2_poi_dataset
        .geometry
        .is_valid
    )
    .sum()
)


notebook_2_compatibility_summary = pd.DataFrame(
    {
        "Check": [
            "Required columns missing",
            "POI categories",
            "Missing category values",
            "Missing geometries",
            "Empty geometries",
            "Invalid geometries",
            "Coordinate reference system",
            "Distance units"
        ],
        "Result": [
            len(missing_notebook_2_columns),
            notebook_2_poi_dataset[
                "poi_category"
            ].nunique(),
            missing_category_values,
            missing_geometries,
            empty_geometries,
            invalid_geometries,
            str(
                notebook_2_poi_dataset.crs
            ),
            "metres"
        ]
    }
)


print("=" * 80)
print("CELL 14 - VERIFY NOTEBOOK 2 COMPATIBILITY")
print("=" * 80)


display(
    notebook_2_compatibility_summary
)


assert not missing_notebook_2_columns

assert missing_category_values == 0

assert missing_geometries == 0

assert empty_geometries == 0

assert invalid_geometries == 0

assert (
    notebook_2_poi_dataset.crs.to_epsg()
    == 32630
)


assert len(
    notebook_2_poi_dataset
) == len(
    combined_poi_dataset
)


assert (
    notebook_2_poi_dataset[
        "poi_category"
    ].nunique()
    ==
    combined_poi_dataset[
        "poi_type"
    ].nunique()
)


print(
    "\nThe POI dataset matches the input "
    "requirements of Notebook 2."
)

CELL 14 - VERIFY NOTEBOOK 2 COMPATIBILITY


,Check,Result
0,Required columns missing,0
1,POI categories,24
2,Missing category values,0
3,Missing geometries,0
4,Empty geometries,0
5,Invalid geometries,0
6,Coordinate reference system,EPSG:32630
7,Distance units,metres



The POI dataset matches the input requirements of Notebook 2.


## What Cell 14 Does

This cell verifies the prepared dataset against the input requirements of `2_Cleaning_and_Feature_Engineering` (Notebook 2) before anything is saved.

It checks that:

- all required fields for `2_Cleaning_and_Feature_Engineering` (Notebook 2) are present;
- the POI categories are complete;
- no category values are missing;
- no geometries are missing;
- no geometries are empty;
- all geometries are valid;
- the coordinate reference system is EPSG:32630;
- spatial distances therefore remain measured in metres;
- the number of observations is unchanged from the validated combined dataset;
- the number of categories is unchanged.

The final file is not written unless these compatibility checks pass.

# 8. Final Output

This section saves the POI dataset prepared for `2_Cleaning_and_Feature_Engineering` (Notebook 2) as the official project GeoPackage and then reloads the saved file for final end-to-end verification.

The official output is:

`data/processed_poi_locations/BT_POI_Dataset.gpkg`

using the layer:

`combined_pois`

This is the POI dataset that `2_Cleaning_and_Feature_Engineering` (Notebook 2) loads when creating location-based POI features.

In [16]:
# ================================================================
# CELL 15 - SAVE THE FINAL POI DATASET
# ================================================================

final_save_start_time = time.time()


temporary_final_poi_file = (
    FINAL_POI_FILE.with_name(
        FINAL_POI_FILE.stem
        +
        ".temporary.gpkg"
    )
)


if temporary_final_poi_file.exists():

    temporary_final_poi_file.unlink()


notebook_2_poi_dataset.to_file(
    temporary_final_poi_file,
    layer=FINAL_POI_LAYER,
    driver="GPKG"
)


assert temporary_final_poi_file.exists()

assert temporary_final_poi_file.stat().st_size > 0


temporary_saved_poi_dataset = gpd.read_file(
    temporary_final_poi_file,
    layer=FINAL_POI_LAYER
)


assert len(
    temporary_saved_poi_dataset
) == len(
    notebook_2_poi_dataset
)


assert temporary_saved_poi_dataset[
    "poi_category"
].nunique() == CONFIGURED_POI_CATEGORY_COUNT


assert list(
    temporary_saved_poi_dataset.columns
) == NOTEBOOK_2_POI_COLUMNS


assert (
    temporary_saved_poi_dataset.crs.to_epsg()
    ==
    32630
)


if FINAL_POI_FILE.exists():

    FINAL_POI_FILE.unlink()


temporary_final_poi_file.replace(
    FINAL_POI_FILE
)


final_save_seconds = (
    time.time()
    -
    final_save_start_time
)


saved_poi_dataset = gpd.read_file(
    FINAL_POI_FILE,
    layer=FINAL_POI_LAYER
)


final_save_summary = pd.DataFrame(
    {
        "Measure": [
            "Final POI file",
            "Final layer",
            "Configured categories",
            "Saved categories",
            "Saved features",
            "Saved columns",
            "CRS",
            "File size (MB)",
            "Saving time (seconds)"
        ],
        "Value": [
            str(
                FINAL_POI_FILE
            ),
            FINAL_POI_LAYER,
            CONFIGURED_POI_CATEGORY_COUNT,
            saved_poi_dataset[
                "poi_category"
            ].nunique(),
            len(
                saved_poi_dataset
            ),
            len(
                saved_poi_dataset.columns
            ),
            str(
                saved_poi_dataset.crs
            ),
            (
                FINAL_POI_FILE.stat().st_size
                /
                1_000_000
            ),
            round(
                final_save_seconds,
                2
            )
        ]
    }
)


print("=" * 80)
print("CELL 15 - SAVE THE FINAL POI DATASET")
print("=" * 80)


display(
    final_save_summary
)


assert FINAL_POI_FILE.exists()

assert FINAL_POI_FILE.stat().st_size > 0


assert len(
    saved_poi_dataset
) == len(
    notebook_2_poi_dataset
)


assert saved_poi_dataset[
    "poi_category"
].nunique() == CONFIGURED_POI_CATEGORY_COUNT


assert list(
    saved_poi_dataset.columns
) == NOTEBOOK_2_POI_COLUMNS


assert saved_poi_dataset.crs.to_epsg() == 32630


print(
    "\nThe final Notebook-2-compatible "
    "POI GeoPackage was saved successfully."
)

CELL 15 - SAVE THE FINAL POI DATASET


,Measure,Value
0,Final POI file,C:\Users\adaml\OneDrive\Desktop\BT_Dissertatio...
1,Final layer,combined_pois
2,Configured categories,24
3,Saved categories,24
4,Saved features,23948
5,Saved columns,8
6,CRS,EPSG:32630
7,File size (MB),4.407296
8,Saving time (seconds),0.46



The final Notebook-2-compatible POI GeoPackage was saved successfully.


## What Cell 15 Does

This cell saves the completed POI dataset as one GeoPackage.

The dataset is first written to a temporary GeoPackage and then reloaded and checked.

The temporary file is accepted only when:

- it exists;
- it has a positive file size;
- the saved row count matches the prepared dataset;
- the saved category count matches the active configuration;
- the final columns are present in the required order;
- the coordinate reference system is EPSG:32630.

Only after these checks pass does the temporary file replace the official final output.

The official output is:

`data/processed_poi_locations/BT_POI_Dataset.gpkg`

using the layer:

`combined_pois`

This safe-write procedure prevents an incomplete write from replacing a valid final POI dataset.

In [17]:
# ================================================================
# CELL 16 - FINAL NOTEBOOK VERIFICATION
# ================================================================

REQUIRED_FINAL_COLUMNS = [
    "poi_id",
    "poi_category",
    "poi_name",
    "osm_element_type",
    "osm_id",
    "poi_source",
    "download_buffer_metres",
    "geometry"
]


final_poi_dataset = gpd.read_file(
    FINAL_POI_FILE,
    layer=FINAL_POI_LAYER
)


final_category_counts = (
    final_poi_dataset.groupby(
        "poi_category",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "Features"
        }
    )
)


final_configured_types = set(
    CONFIGURED_POI_TYPES
)


final_observed_types = set(
    final_poi_dataset[
        "poi_category"
    ].unique()
)


final_checks = pd.DataFrame(
    {
        "Check": [
            "Every configured category download is valid",
            "Final GeoPackage exists",
            "Final GeoPackage size is positive",
            "Expected final columns present",
            "Configured and final categories match",
            "All configured POI categories present",
            "POI identifiers complete",
            "POI identifiers unique",
            "POI categories complete",
            "POI names complete",
            "Geometries complete",
            "All geometries are points",
            "Projected CRS is EPSG:32630",
            "POI source is recorded",
            "5 km POI download buffer is recorded"
        ],
        "Result": [
            bool(
                poi_download_inventory[
                    "Valid Download"
                ].all()
            ),

            FINAL_POI_FILE.exists(),

            (
                FINAL_POI_FILE.stat().st_size
                >
                0
            ),

            (
                list(
                    final_poi_dataset.columns
                )
                ==
                REQUIRED_FINAL_COLUMNS
            ),

            (
                final_configured_types
                ==
                final_observed_types
            ),

            (
                final_poi_dataset[
                    "poi_category"
                ].nunique()
                ==
                CONFIGURED_POI_CATEGORY_COUNT
            ),

            final_poi_dataset[
                "poi_id"
            ].notna().all(),

            final_poi_dataset[
                "poi_id"
            ].is_unique,

            final_poi_dataset[
                "poi_category"
            ].notna().all(),

            final_poi_dataset[
                "poi_name"
            ].notna().all(),

            final_poi_dataset.geometry.notna().all(),

            bool(
                (
                    final_poi_dataset.geometry.geom_type
                    ==
                    "Point"
                ).all()
            ),

            (
                final_poi_dataset.crs.to_epsg()
                ==
                32630
            ),

            bool(
                (
                    final_poi_dataset[
                        "poi_source"
                    ]
                    ==
                    "OpenStreetMap Overpass"
                ).all()
            ),

            bool(
                (
                    final_poi_dataset[
                        "download_buffer_metres"
                    ]
                    ==
                    POI_DOWNLOAD_BUFFER_METRES
                ).all()
            )
        ]
    }
)


final_notebook_summary = pd.DataFrame(
    {
        "Measure": [
            "Original study xmin",
            "Original study ymin",
            "Original study xmax",
            "Original study ymax",
            "POI download buffer (metres)",
            "Configured POI categories",
            "Final POI categories",
            "Final POI features",
            "Final POI columns",
            "Final file size (MB)",
            "Final file",
            "Final layer"
        ],
        "Value": [
            STUDY_XMIN,
            STUDY_YMIN,
            STUDY_XMAX,
            STUDY_YMAX,
            POI_DOWNLOAD_BUFFER_METRES,
            CONFIGURED_POI_CATEGORY_COUNT,
            final_poi_dataset[
                "poi_category"
            ].nunique(),
            len(
                final_poi_dataset
            ),
            len(
                final_poi_dataset.columns
            ),
            (
                FINAL_POI_FILE.stat().st_size
                /
                1_000_000
            ),
            str(
                FINAL_POI_FILE
            ),
            FINAL_POI_LAYER
        ]
    }
)


incomplete_temporary_category_files = list(
    POI_DOWNLOAD_FOLDER.glob(
        "*.temporary.gpkg"
    )
)


incomplete_final_files = list(
    PROCESSED_POI_FOLDER.glob(
        "*.temporary.gpkg"
    )
)


print("=" * 80)
print("CELL 16 - FINAL NOTEBOOK VERIFICATION")
print("=" * 80)


display(
    final_checks
)


print(
    "\nFinal notebook summary:"
)


display(
    final_notebook_summary
)


print(
    "\nFinal features by POI category:"
)


display(
    final_category_counts
)


print(
    f"\nIncomplete category write files: "
    f"{len(incomplete_temporary_category_files):,}"
)


print(
    f"Incomplete final write files: "
    f"{len(incomplete_final_files):,}"
)


assert final_checks[
    "Result"
].all()


assert len(
    incomplete_temporary_category_files
) == 0


assert len(
    incomplete_final_files
) == 0


assert list(
    final_poi_dataset.columns
) == NOTEBOOK_2_POI_COLUMNS


assert len(
    final_poi_dataset
) == len(
    notebook_2_poi_dataset
)


print(
    "\nNotebook 1 is complete."
)


print(
    "\nThe original BT study area was retained."
)


print(
    "The 5 km buffer was used only for the "
    "Point of Interest downloads."
)


print(
    "\nThe final POI dataset is fully compatible "
    "with Notebook 2."
)


print(
    "\nNotebook 2 input:"
)


print(
    FINAL_POI_FILE
)

CELL 16 - FINAL NOTEBOOK VERIFICATION


,Check,Result
0,Every configured category download is valid,True
1,Final GeoPackage exists,True
2,Final GeoPackage size is positive,True
3,Expected final columns present,True
4,Configured and final categories match,True
5,All configured POI categories present,True
6,POI identifiers complete,True
7,POI identifiers unique,True
8,POI categories complete,True
9,POI names complete,True



Final notebook summary:


,Measure,Value
0,Original study xmin,530000
1,Original study ymin,5590000
2,Original study xmax,668000
3,Original study ymax,5650000
4,POI download buffer (metres),5000
5,Configured POI categories,24
6,Final POI categories,24
7,Final POI features,23948
8,Final POI columns,8
9,Final file size (MB),4.407296



Final features by POI category:


,poi_category,Features
0,airports,27
1,attractions,302
2,beaches,481
3,bus_stations,23
4,bus_stops,11117
5,cafes,1704
6,fast_foods,1632
7,gyms,136
8,holiday_parks,195
9,hospitals,78



Incomplete category write files: 0
Incomplete final write files: 0

Notebook 1 is complete.

The original BT study area was retained.
The 5 km buffer was used only for the Point of Interest downloads.

The final POI dataset is fully compatible with Notebook 2.

Notebook 2 input:
C:\Users\adaml\OneDrive\Desktop\BT_Dissertation_AL\data\processed_poi_locations\BT_POI_Dataset.gpkg


## What Cell 16 Does

This cell reloads the official saved GeoPackage and performs the final end-to-end notebook verification.

It confirms that:

- every category currently defined in `POI_CONFIG` has a valid individual download;
- the final GeoPackage exists and has a positive file size;
- the final columns are present in the exact required order;
- the configured and saved category sets match;
- all configured POI categories are present;
- all POI identifiers are complete and unique;
- all POI categories are complete;
- all POI names are complete;
- all geometries are complete;
- every geometry is a point;
- the final dataset uses EPSG:32630;
- the OpenStreetMap Overpass source is recorded;
- the 5 km POI download buffer is recorded;
- the saved dataset matches the prepared dataset for `2_Cleaning_and_Feature_Engineering` (Notebook 2);
- no incomplete temporary category or final-output files remain.

The category count remains dynamic and is calculated from the active `POI_CONFIG`.

Passing this cell confirms that `1_POI_Dataset_Overpass` (Notebook 1) has successfully produced the official POI input for `2_Cleaning_and_Feature_Engineering` (Notebook 2).

# Notebook Completion

`1_POI_Dataset_Overpass` (Notebook 1) is complete.

The original BT study area has been retained using the supplied UTM Zone 30N coordinates:

- xmin = 530000
- ymin = 5590000
- xmax = 668000
- ymax = 5650000

A 5 km buffer was used only for collecting nearby OpenStreetMap POIs. It does not alter the original BT analysis boundary.

All active POI categories were downloaded directly from OpenStreetMap through the Overpass workflow, verified, standardised and combined.

The combined dataset was then converted to the exact schema required by `2_Cleaning_and_Feature_Engineering` (Notebook 2) and independently checked before and after saving.

The official output is:

`data/processed_poi_locations/BT_POI_Dataset.gpkg`

using the layer:

`combined_pois`

The final schema is:

- `poi_id`
- `poi_category`
- `poi_name`
- `osm_element_type`
- `osm_id`
- `poi_source`
- `download_buffer_metres`
- `geometry`

The dataset uses EPSG:32630, allowing the downstream notebooks to calculate POI distances in metres.

This GeoPackage is now the official Point of Interest input for `2_Cleaning_and_Feature_Engineering` (Notebook 2). The POI information created from it is then carried forward into `3_Model_1` (Notebook 3) and `4_Model_2` (Notebook 4).